# Test EMDB dataset harvesting code

## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sindex.sources.emdb.client import get_emdb_id_record
from sindex.sources.emdb.normalize import slim_emdb_record
from sindex.sources.emdb.jobs import (
    harvest_emdb_datasets_for_date_range_to_ndjson,
    batch_slim_emdb_record_to_ndjson
    )
from pathlib import Path

## Get record for single EMDB ID and create slim metadata

In [3]:
emdb_id = "EMD-24511"

rec = get_emdb_id_record(emdb_id)

if rec is None:
    print("No record found.")
else:
    print("Success")
    display(rec)

Success


{'_id': '6357ba3fd79f45429486119f',
 'admin': {'authors_list': {'author': [{'instance_type': 'author',
     'valueOf_': 'Wei X'},
    {'instance_type': 'author', 'valueOf_': 'Marmorstein R'}]},
  'current_status': {'code': {'valueOf_': 'REL'},
   'date': '2025-05-21T00:00:00',
   'processing_site': 'RCSB'},
  'grant_support': {'grant_reference': [{'country': 'United States',
     'funding_body': 'National Institutes of Health/National Institute of General Medical Sciences (NIH/NIGMS)',
     'instance_type': 'grant_reference'}]},
  'key_dates': {'deposition': '2021-07-22T00:00:00',
   'header_release': '2023-05-10T00:00:00',
   'map_release': '2023-05-10T00:00:00',
   'update': '2025-05-21T00:00:00'},
  'keywords': 'mutant, TRANSFERASE',
  'revision_history': {'revision': [{'change_list': {'revision_change_sub_group': [{'instance_type': 'image',
        'provider': 'REPOSITORY',
        'revision_type': 'INITIAL_RELEASE'},
       {'instance_type': 'primary_map',
        'provider': 'REP

In [4]:
slim_emdb_record(rec)

{'source': 'emdb',
 'identifiers': [{'identifier': 'EMD-24511', 'identifier_type': 'emdb_id'}],
 'url': 'https://www.ebi.ac.uk/emdb/EMD-24511',
 'title': 'Structure of ACLY D1026A-substrates-asym-int',
 'subjects': ['mutant', 'TRANSFERASE'],
 'publication_date': '2021-07-22T00:00:00',
 'creators': [{'name': 'Wei X',
   'name_type': 'Personal',
   'identifiers': ['https://orcid.org/0000-0002-6514-7076']},
  {'name': 'Marmorstein R',
   'name_type': 'Personal',
   'identifiers': ['https://orcid.org/0000-0003-4373-4752']}],
 'publisher': 'The Electron Microscopy Data Bank (EMDB)'}

In [6]:
emdb_id = "sdgdsg" #wrong EMDB id should give error

rec = get_emdb_id_record(emdb_id)

if rec is None:
    print("No record found.")
else:
    print("Success")
    display(rec)

ValueError: Invalid EMDB ID: sdgdsg

## Write records to ndjson for date range

In [7]:
start_date_str = "2014-01-01"
end_date_str = "2014-12-31"

out_dir = Path("output/emdb/harvest_date_range_test")
out_dir.mkdir(parents=True, exist_ok=True)

n = harvest_emdb_datasets_for_date_range_to_ndjson(
    start_date_str = start_date_str,
    end_date_str = end_date_str,
    save_folder = out_dir
)

print(f"Done. Wrote {n:,} records {out_dir.resolve()}")

Fetching EMDB IDs from CSV search endpoint...
Fetched 53022 EMDB IDs.
Harvesting entries from 2014-01-01 to 2014-12-31 using 12 workers...
Processed 53022/53022 IDs (written: 662)

Done. Wrote 662 entries to output\emdb\harvest_date_range_test\emdb_deposited_2014-01-01_to_2014-12-31.ndjson
Done. Wrote 662 records C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\output\emdb\harvest_date_range_test


## Write batch slim records

In [8]:
# Folder containing raw DataCite NDJSON or NDJSON.GZ files
src_folder = Path("output/emdb/harvest_date_range_test")

# Folder where slimmed outputs will be written
dst_folder = Path("output/emdb/batch_slim_test")

# Safety checks
if not src_folder.exists():
    raise FileNotFoundError(
        f"Source folder does not exist: {src_folder.resolve()}"
    )

dst_folder.mkdir(parents=True, exist_ok=True)

print("Running EMDB batch slimming test")
print(f"  Input folder : {src_folder.resolve()}")
print(f"  Output folder: {dst_folder.resolve()}")
print("---")

# Run the batch slim function
summary = batch_slim_emdb_record_to_ndjson(
    src_folder=str(src_folder),
    dst_folder=str(dst_folder),
    overwrite=True,          # overwrite for repeatable tests
    accept_gz=True,
    one_line_progress=False, # easier to read in logs
)

# Summary
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\nTest completed successfully.")

Running EMDB batch slimming test
  Input folder : C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\output\emdb\harvest_date_range_test
  Output folder: C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\output\emdb\batch_slim_test
---
Done. files=1 kept=662 bad=0 time=0.3s rate≈2,279/s → C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\output\emdb\batch_slim_test

Summary:
  files_seen: 1
  records_read: 662
  records_kept: 662
  records_bad_json: 0
  output_dir: C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\output\emdb\batch_slim_test
  elapsed_sec: 0.29
  rate_rec_per_sec: 2279

Test completed successfully.
